# Tutorial 10 - Convolutional Neural Networks

## Dr. David C. Schedl

This tutorial is geared towards students **experienced in programming** and aims to introduce you to **Digital Imaging / Computer Vision** techniques.
More precisely, it covers **Convolutional Neural Networks (CNNs)** in **PyTorch**.

We build on the previous tutorial (`09_NNs`): we reuse the same **CIFAR10** dataset and the same **training loop**, but replace the linear/MLP model with a classic **LeNet** CNN.

For training, it is recommended to use a **GPU**. In Google Colab go to the menu and select **Runtime** -> **Change runtime type** -> **Hardware accelerator** -> switch to **GPU**.

## Setup

Let's load and preprocess the **CIFAR10** dataset exactly as in the previous tutorial.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

batch_size = 100

# Load and preprocess the CIFAR10 dataset
transform = transforms.Compose([transforms.ToTensor(),
                              transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False)

# the labels will be put in a separate vector as the original is just numbers, but we want text labels
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

# display some images
import matplotlib.pyplot as plt
import numpy as np

# functions to show an image
def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# get some random training images
dataiter = iter(trainloader)
images, labels = next(dataiter)

# show 10 images
plt.title('|'.join('% 5s' % classes[labels[j].item()] for j in range(10)))
imshow(torchvision.utils.make_grid(images[:10], nrow=10))

## The Model: LeNet

Instead of a linear classifier or an MLP, we now use a **convolutional neural network**. We implement a classic **LeNet**-style architecture, adapted for the CIFAR10 input size of $3\times32\times32$.

| Layer                     | Output Size  |
| ------------------------- | ------------ |
| Input                     | 3 x 32 x 32  |
| Conv (Cout=6, K=5) + ReLU | 6 x 28 x 28  |
| MaxPool (K=2, S=2)        | 6 x 14 x 14  |
| Conv (Cout=16, K=5) + ReLU| 16 x 10 x 10 |
| MaxPool (K=2, S=2)        | 16 x 5 x 5   |
| Flatten                   | 400          |
| Linear (400 -> 120) + ReLU| 120          |
| Linear (120 -> 84) + ReLU | 84           |
| Linear (84 -> 10)         | 10           |

Note how the convolutional layers operate directly on the 2D image (keeping the spatial structure), unlike the linear model which flattened the image right away.

In [ ]:
# Define the LeNet CNN
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, kernel_size=5)   # 3x32x32 -> 6x28x28
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)  # 6x14x14 -> 16x10x10
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)  # flatten all dimensions except the batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# print number of parameters
model = LeNet()
print(model)
print('parameters:', sum(p.numel() for p in model.parameters()))

## Training

We train exactly as in the previous tutorial: a **cross-entropy loss** and the **Adam** optimizer. We loop over the training dataset for a number of epochs, feed the images and labels to the model, compute the loss, and backpropagate to update the model's parameters.

Afterwards we evaluate on the test set inside a `torch.no_grad()` context (no gradients needed for testing).

In [ ]:
model = LeNet()

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
#optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Train the model
for epoch in range(10):  # loop over the dataset multiple times
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # keep loss for statistics
        running_loss += loss.item()
    # print statistics
    print('Epoch %d loss: %.3f' % (epoch + 1, running_loss / len(trainloader)))

print('Finished Training')

# Test the model
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 10000 test images: %d %%' % (100 * correct / total))

Let's look at a few images from the test set and print the model's predictions.

In [ ]:
# get some random test images
images, labels = next(iter(testloader))


# set up a figure
fig = plt.figure(figsize=(15, 7))
fig.subplots_adjust(left=0, right=1, bottom=0, top=1, hspace=0.05, wspace=0.05)

_, prediction_label = torch.max(model(images).data, 1)

total, correct = 0, 0
# plot the images
for i,img in enumerate(images[:50]):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    ax = fig.add_subplot(5, 10, i + 1, xticks=[], yticks=[])
    ax.imshow(np.transpose(npimg, (1, 2, 0)), interpolation='nearest')


    img_text = f'{classes[prediction_label[i]]} [{classes[labels[i]]}]'

    if prediction_label[i] == labels[i]:
        # label the image with the green text
        ax.text(0.1, 0.1, img_text, color='lightgreen', transform=ax.transAxes)
        ax.tick_params(color='green', labelcolor='green')
        for spine in ax.spines.values():
            spine.set_edgecolor('green')
        correct += 1
    else:
        # label the image with the red text
        ax.text(0.1, 0.1, img_text, color='darkred', transform=ax.transAxes)
        ax.tick_params(color='red', labelcolor='red')
        for spine in ax.spines.values():
            spine.set_edgecolor('red')
    total += 1

print(f'Accuracy: {correct/total*100:.2f}% for {total} test images')

### Exercise 01 📝: Compare the CNN to the linear / MLP model

In the previous tutorial (`09_NNs`) we classified CIFAR10 with a linear model (and an MLP in the exercise). How does the LeNet CNN compare in terms of accuracy?

- How many parameters does the CNN have compared to the linear model?
- Try adding more convolutional filters or another conv layer. Does accuracy improve?
- What happens if you train for more epochs?